# GAN 50장 생성과 분류기 예비 비교

기존 정상·언더필 사진을 이용해 **실제 언더필 50장 → DCGAN 생성 50장**을 시험합니다. 같은 실제 시험 사진으로 ResNet18 세 조건을 비교합니다.

1. **실제만:** 실제 언더필 50장
2. **GAN 추가:** 같은 실제 50장 + GAN 50장
3. **좌우 반전 추가:** 같은 실제 50장 + 반전한 50장 (보조 대조)

이것은 한 번의 분할·한 시드 **예비 실험**입니다. 원본은 클래스별 출력 1회이며 fold 0도 기존 연구에서 분석한 적이 있으므로, 이번 학습에서 제외했어도 독립적인 새 출력의 최종 성능이 아닙니다. 생성 이미지와 자동 선택한 실제 사진은 육안 검토가 필요합니다.

**Drive `내 드라이브/산공종설/`에 필요한 파일**

- `underfill_originals_4000.zip` — 기존 원본 4,000장
- `dinov2_grid_bundle_20261013.zip` — 기존 분할·선별·베드 지도
- `gan50_code_bundle_20261101.zip` — 이 노트북과 함께 받은 작은 코드 묶음

Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 선택하고 셀을 순서대로 실행하세요. GPU 학습이 실제로 시작되며 사용량이 발생합니다. 패키지를 새로 설치하지 않지만, ResNet18의 ImageNet 사전학습 가중치는 처음 실행 시 네트워크에서 내려받을 수 있습니다.


## 1. Drive 연결과 실행 설정

기본값은 이해를 위한 첫 시험값입니다. GAN 1,500회와 분류기 400회 업데이트가 최적이라는 근거는 없습니다. `RUN_NAME`을 바꾸면 새로운 실험 폴더가 만들어집니다. **같은 폴더에서 중단 복구할 때는 설정을 바꾸지 마세요.**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, csv, hashlib, time, zipfile, subprocess, shlex, shutil, tempfile
import numpy as np
from PIL import Image, ImageDraw, ImageOps
import torch, torchvision
from IPython.display import display

DRIVE_DIR = Path('/content/drive/MyDrive/산공종설')
ORIGINAL_ZIP = DRIVE_DIR / 'underfill_originals_4000.zip'
GRID_ZIP = DRIVE_DIR / 'dinov2_grid_bundle_20261013.zip'
CODE_ZIP = DRIVE_DIR / 'gan50_code_bundle_20261101.zip'
RUN_NAME = 'gan50_20261101_run01'
SEED = 42
GAN_STEPS = 1500
GAN_BATCH = 8
GAN_SAVE_EVERY = 100
CLASSIFIER_STEPS = 400
CLASSIFIER_BATCH = 16
AUTO_DOWNLOAD = True

RUN_ROOT = DRIVE_DIR / 'results' / RUN_NAME
PREPARED = RUN_ROOT / 'prepared'
DATA = PREPARED / 'data'
GAN_DIR = RUN_ROOT / 'gan'
LOG_DIR = RUN_ROOT / 'logs'
PREVIEW_DIR = RUN_ROOT / 'review'
WORK_CODE = Path('/content/gan50_code_20261101')

if not torch.cuda.is_available():
    raise RuntimeError('GPU가 연결되지 않았습니다. 런타임 유형을 GPU로 바꿔 주세요.')
for required in (ORIGINAL_ZIP, GRID_ZIP, CODE_ZIP):
    if not required.is_file():
        raise FileNotFoundError(f'필요한 Drive 파일이 없습니다: {required}')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('결과 폴더:', RUN_ROOT)


## 2. 입력 파일과 설정을 고정합니다

파일의 지문(SHA-256)을 기록해, 중단 후 서로 다른 사진이나 코드가 섞이지 않도록 합니다. 원본 ZIP이 약 1GB이므로 이 확인은 잠시 걸릴 수 있습니다.


In [ ]:
def hash_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

EXPECTED_CODE_SHA256 = '0cee823e2c5dd761f1cd810444607338607bbffbc5d2a18e5dc1ab98f3efcc94'
actual_code_sha = hash_file(CODE_ZIP)
if actual_code_sha != EXPECTED_CODE_SHA256:
    raise RuntimeError('코드 ZIP 지문이 이 노트북과 다릅니다. 같은 묶음의 ZIP을 다시 올려 주세요.')

run_lock = {
    'original_zip_sha256': hash_file(ORIGINAL_ZIP),
    'grid_zip_sha256': hash_file(GRID_ZIP),
    'code_zip_sha256': actual_code_sha,
    'seed': SEED, 'gan_steps': GAN_STEPS, 'gan_batch': GAN_BATCH,
    'gan_save_every': GAN_SAVE_EVERY,
    'classifier_steps': CLASSIFIER_STEPS, 'classifier_batch': CLASSIFIER_BATCH,
    'created_for': '50 real → 50 generated; single time-block pilot',
}
RUN_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)
PREVIEW_DIR.mkdir(exist_ok=True)
lock_file = RUN_ROOT / 'run_lock.json'
if lock_file.exists():
    if json.loads(lock_file.read_text(encoding='utf-8')) != run_lock:
        raise RuntimeError('기존 실험과 입력/설정이 다릅니다. RUN_NAME을 새 이름으로 바꿔 주세요.')
else:
    lock_file.write_text(json.dumps(run_lock, ensure_ascii=False, indent=2), encoding='utf-8')

WORK_CODE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as archive:
    expected_names = {'prepare_data.py', 'gan_model.py', 'classifier_compare.py'}
    if set(archive.namelist()) != expected_names:
        raise ValueError('코드 ZIP 파일 목록이 예상과 다릅니다.')
    for name in sorted(expected_names):
        (WORK_CODE / name).write_bytes(archive.read(name))
print('입력과 설정 확인 완료. 같은 설정으로 중단 복구할 수 있습니다.')


### 입력 ZIP을 임시 저장소에 복사

원본 ZIP을 반복해서 읽을 때 Drive 접속이 느려질 수 있어, Colab 임시 저장소 `/content`에 한 번 복사합니다. 원본 Drive 파일은 그대로 둡니다. 런타임이 종료되면 임시 복사본은 사라지며, 재개 시 다시 복사합니다.


In [ ]:
INPUT_CACHE = Path('/content/gan50_inputs')
INPUT_CACHE.mkdir(parents=True, exist_ok=True)

def stage_input(source, fingerprint, label):
    target = INPUT_CACHE / f'{label}_{fingerprint[:16]}.zip'
    if target.exists():
        if hash_file(target) != fingerprint:
            raise RuntimeError(f'기존 임시 파일의 지문이 다릅니다: {target}')
        print('임시 복사본 재사용:', target)
        return target
    print('임시 저장소에 복사 중:', label, flush=True)
    handle, temp_name = tempfile.mkstemp(prefix=f'{label}_', suffix='.partial', dir=INPUT_CACHE)
    os.close(handle)
    temp_path = Path(temp_name)
    try:
        shutil.copyfile(source, temp_path)
        if hash_file(temp_path) != fingerprint:
            raise RuntimeError('임시 복사본의 지문이 원본과 다릅니다.')
        os.replace(temp_path, target)
    finally:
        if temp_path.exists():
            temp_path.unlink()
    return target

LOCAL_ORIGINAL = stage_input(ORIGINAL_ZIP, run_lock['original_zip_sha256'], 'original')
LOCAL_GRID = stage_input(GRID_ZIP, run_lock['grid_zip_sha256'], 'grid')
print('임시 입력 준비 완료')


## 3. 로그를 Drive에 저장하는 실행 도구

진행 상태를 짧게 표시하고 전체 출력은 Drive의 `logs/`에 저장합니다. 오류가 나면 다음 셀로 자동 진행하지 않습니다.


In [ ]:
def run_logged(command, log_path):
    print('실행:', ' '.join(shlex.quote(str(x)) for x in command))
    with Path(log_path).open('a', encoding='utf-8') as log:
        log.write('\n--- ' + time.strftime('%Y-%m-%d %H:%M:%S') + ' ---\n')
        log.flush()
        proc = subprocess.Popen([str(x) for x in command], stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                                errors='replace', bufsize=1)
        try:
            for line in proc.stdout:
                log.write(line)
                log.flush()
                print(line, end='', flush=True)
            status = proc.wait()
        except BaseException:
            proc.terminate()
            raise
    if status:
        raise RuntimeError(f'실행 실패({status}). Drive 로그 확인: {log_path}')


## 4. 실제 사진 선별과 분할

- 기존 선별표(v3.2)를 사용하고 **원본 512×512 표면 후보 → 256×256**으로 준비합니다. 베드 지도는 위치를 고르는 데만 사용하고, 사진을 회색으로 칠하지 않습니다.
- 원본 기준 fold 0은 이번 예비 시험, fold 4는 내부 검증, 인접 30프레임은 제외합니다. 그 나머지 학습 구간에서 정상 최대 500장과 언더필 50장을 선택합니다.
- 마스크의 표면 추정 비율 **99.5%는 임시 자동 기준**입니다. 정답 마스크가 아니며, 언더필 출력 조건 라벨만으로 모든 사진에 눈에 보이는 결함이 있다고 보장하지 못합니다. 아래 50장 확인판을 꼭 살펴봐 주세요.
- 중단으로 준비가 끝나지 않았다면 기존 폴더를 지우지 말고 `RUN_NAME`을 새 이름으로 바꿔 처음부터 다시 실행합니다.


In [ ]:
audit_path = PREPARED / 'audit_settings.json'
if audit_path.exists():
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    if audit.get('smoke_test') or audit.get('seed') != SEED:
        raise RuntimeError('기존 준비 결과의 시드/시험 설정이 다릅니다. 새 RUN_NAME을 사용하세요.')
    print('기존 사진 준비 결과를 재사용합니다:', PREPARED)
else:
    if PREPARED.exists() and any(PREPARED.iterdir()):
        raise RuntimeError('사진 준비가 중간에 멈췄습니다. 삭제하지 말고 새 RUN_NAME을 사용하세요.')
    run_logged([sys.executable, WORK_CODE / 'prepare_data.py',
                '--original-zip', LOCAL_ORIGINAL, '--bundle', LOCAL_GRID,
                '--out', PREPARED, '--seed', SEED], LOG_DIR / 'prepare.log')
audit = json.loads(audit_path.read_text(encoding='utf-8'))
print('선택 수:', audit['train_defect_count'], '언더필 /', audit['train_normal_count'], '정상')
print('표면 선택은 자동 추정이며 사람의 육안 확인이 필요합니다.')


## 5. GAN에 넣을 실제 언더필 50장 보기

사진 제목의 앞 8자리는 원본 프레임 번호입니다. 흐림·과노출, 언더필 틈, 베드/회색 경계가 섞였는지 살펴봐 주세요. **자동으로 다음 셀을 막지 않으므로** `런타임 → 모두 실행`을 선택했다면 작업이 계속됩니다. 눈으로 확인하려면 여기까지 실행한 뒤 다음 셀을 진행하세요.


In [ ]:
def image_paths(folder):
    return sorted((p for p in Path(folder).iterdir()
                   if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg'}),
                  key=lambda p: p.name)

def make_sheet(paths, dest, title_prefix, cols=5, thumb=176, caption=28):
    rows = (len(paths) + cols - 1) // cols
    sheet = Image.new('RGB', (cols * thumb, rows * (thumb + caption)), 'white')
    draw = ImageDraw.Draw(sheet)
    for i, path in enumerate(paths):
        with Image.open(path) as original:
            picture = ImageOps.fit(original.convert('RGB'), (thumb, thumb), method=Image.Resampling.LANCZOS)
        x, y = (i % cols) * thumb, (i // cols) * (thumb + caption)
        sheet.paste(picture, (x, y))
        draw.text((x + 4, y + thumb + 4), f'{i+1:02d}  {path.stem[:18]}', fill='black')
    sheet.save(dest)
    print(title_prefix, ':', dest)
    display(sheet.resize((min(sheet.width, 800), round(sheet.height * min(sheet.width, 800) / sheet.width))))

real_paths = image_paths(DATA / 'train_real' / 'underfill')
if len(real_paths) != 50:
    raise RuntimeError(f'GAN 입력이 50장이 아닙니다: {len(real_paths)}')
make_sheet(real_paths, PREVIEW_DIR / 'real_underfill_50.png', '실제 학습 사진 50장')


## 6. 실제 50장으로 GAN을 학습하고 50장 생성

입문용 DCGAN으로 작은 파일럿을 합니다. 결과가 실제 결함을 정확히 재현하거나 유용하다는 보장은 없습니다. 100회마다 Drive에 체크포인트를 저장합니다. Colab 연결이 끊기면 **같은 노트북·같은 RUN_NAME·같은 설정**으로 위 셀부터 다시 실행하면 마지막 저장 시점부터 이어갑니다. 저장 전 작업은 다시 수행됩니다.


In [ ]:
run_logged([sys.executable, WORK_CODE / 'gan_model.py',
            '--data', DATA / 'train_real' / 'underfill', '--out', GAN_DIR,
            '--steps', GAN_STEPS, '--seed', SEED, '--num-generate', 50,
            '--device', 'cuda', '--batch', GAN_BATCH,
            '--save-every', GAN_SAVE_EVERY, '--log-every', 25], LOG_DIR / 'gan.log')

gan_progress = json.loads((GAN_DIR / 'progress.json').read_text(encoding='utf-8'))
print('GAN 상태:', gan_progress['status'], '단계:', gan_progress['step'])


## 7. 생성 50장 보기와 원본과의 단순 비교

**전체 50장을 빠짐없이** 저장합니다. 각 생성 사진과 비슷한 학습 사진을 64×64 픽셀의 단순 차이로 하나씩 찾아 나란히 표시합니다. 조명·위치 변화에 민감한 대략적인 확인일 뿐, 복사 여부나 품질을 증명하지 않습니다. 학습 사진도 실제 결함이 선명한지 사람이 확인해야 합니다.


In [ ]:
fake_paths = image_paths(GAN_DIR / 'generated')
expected_names = [f'generated_{i:03d}.png' for i in range(1, 51)]
if [p.name for p in fake_paths] != expected_names:
    raise RuntimeError('생성 PNG가 예상한 50장과 다릅니다. GAN 로그와 progress.json을 확인하세요.')
make_sheet(fake_paths, PREVIEW_DIR / 'generated_50.png', 'GAN 생성 사진 50장')

def pixel_view(path):
    with Image.open(path) as image:
        return np.asarray(image.convert('RGB').resize((64, 64), Image.Resampling.BILINEAR),
                          dtype=np.float32) / 255.0

real_small = np.stack([pixel_view(path) for path in real_paths])
nearest = []
for fake in fake_paths:
    image = pixel_view(fake)
    errors = ((real_small - image) ** 2).mean(axis=(1, 2, 3))
    idx = int(errors.argmin())
    nearest.append({'generated': fake.name, 'nearest_training_image': real_paths[idx].name,
                    'pixel_mse_64': float(errors[idx]), 'fake_path': fake,
                    'real_path': real_paths[idx]})
with (PREVIEW_DIR / 'nearest_training_pairs.csv').open('w', encoding='utf-8-sig', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=['generated', 'nearest_training_image', 'pixel_mse_64'], extrasaction='ignore')
    writer.writeheader()
    writer.writerows(nearest)

for group in range(5):
    pairs = nearest[group * 10:(group + 1) * 10]
    side, cap = 160, 26
    sheet = Image.new('RGB', (side * 2, len(pairs) * (side + cap)), 'white')
    draw = ImageDraw.Draw(sheet)
    for row, pair in enumerate(pairs):
        y = row * (side + cap)
        for col, key in enumerate(('fake_path', 'real_path')):
            with Image.open(pair[key]) as original:
                sheet.paste(ImageOps.fit(original.convert('RGB'), (side, side), method=Image.Resampling.LANCZOS), (col * side, y))
        draw.text((3, y + side + 4), pair['generated'][:20], fill='black')
        draw.text((side + 3, y + side + 4), pair['nearest_training_image'][:20], fill='black')
    dest = PREVIEW_DIR / f'nearest_pairs_{group+1:02d}.png'
    sheet.save(dest)
print('비슷한 학습 사진 비교: 5개 파일 × 10쌍. 왼쪽 생성 / 오른쪽 학습 사진')
display(Image.open(PREVIEW_DIR / 'nearest_pairs_01.png'))


## 8. 분류기 실행 전: GAN에 실제로 들어간 50장 재확인

준비한 사진 50장의 파일명과 SHA-256이 GAN의 학습 목록과 같은지, GAN이 끝났는지와 생성 이미지가 정확히 50장인지 확인합니다. 사진이 바뀌면 실행을 멈춥니다.


In [ ]:
gan_progress = json.loads((GAN_DIR / 'progress.json').read_text(encoding='utf-8'))
gan_manifest = json.loads((GAN_DIR / 'training_manifest.json').read_text(encoding='utf-8'))
gan_config = json.loads((GAN_DIR / 'gan_config.json').read_text(encoding='utf-8'))
generation_info = json.loads((GAN_DIR / 'generation_info.json').read_text(encoding='utf-8'))
saved = {row['filename']: row['sha256'] for row in gan_manifest['images']}
current = {path.name: hash_file(path) for path in real_paths}
if len(saved) != 50 or len(current) != 50 or saved != current:
    raise RuntimeError('GAN의 입력 50장과 현재 분류기 실사 50장이 다릅니다. 새 RUN_NAME으로 준비하세요.')
if gan_manifest['fingerprint'] != gan_config['dataset_fingerprint']:
    raise RuntimeError('GAN 학습 목록의 데이터 지문이 설정과 다릅니다.')
if gan_progress['status'] != 'completed' or gan_progress['step'] != GAN_STEPS:
    raise RuntimeError('GAN 학습이 완료되지 않았거나 요청한 단계와 다릅니다.')
if generation_info['checkpoint_step'] != GAN_STEPS or generation_info['num_generate'] != 50:
    raise RuntimeError('생성 이미지의 체크포인트/개수가 현재 설정과 다릅니다.')
if len(image_paths(GAN_DIR / 'generated')) != 50:
    raise RuntimeError('생성 이미지가 정확히 50장이 아닙니다.')
print('확인 완료: 같은 실제 언더필 50장, GAN 완료, 생성 이미지 50장')


## 9. ResNet18 세 조건 비교

각 조건에서 같은 정상 사진, 같은 실제 언더필 50장, 같은 평가 사진과 **동일 업데이트 횟수**를 사용합니다. GAN 추가와 반전 추가는 언더필 샘플 중 실제와 추가 이미지를 각각 절반씩 뽑습니다. 따라서 실제만 사용한 조건보다 **실제 언더필 사진을 본 횟수가 적어집니다.** 이 비교는 학습량을 고정한 혼합 효과이며, 실제 사진 노출을 그대로 유지한 채 합성만 더한 효과는 아닙니다.

좌우 반전이 실제 공정 방향상 타당한지는 아직 확인되지 않았으므로 세 번째 조건은 보조 대조입니다. 기준값 0.5와 마지막 단계 사용은 이 예비 시험에서 미리 고정합니다. 예전 분류기를 그대로 가져오지 않고 동일한 ImageNet 사전학습 가중치에서 세 조건을 학습합니다.

**중단 복구:** GAN과 달리 분류기는 중간 복구를 지원하지 않습니다. 다시 실행하면 기존 결과를 건드리지 않고 새 `classifier_NNN` 폴더에 처음부터 학습합니다.


In [ ]:
existing = [p for p in RUN_ROOT.glob('classifier_*') if p.is_dir() and p.name[11:].isdigit()]
next_no = max([int(p.name[11:]) for p in existing], default=0) + 1
CLASSIFIER_DIR = RUN_ROOT / f'classifier_{next_no:03d}'
run_logged([sys.executable, WORK_CODE / 'classifier_compare.py',
            '--data', DATA, '--gan-dir', GAN_DIR / 'generated', '--out', CLASSIFIER_DIR,
            '--steps', CLASSIFIER_STEPS, '--seed', SEED, '--device', 'cuda',
            '--batch-size', CLASSIFIER_BATCH, '--size', 256,
            '--normal-max', 500, '--expected-underfill', 50, '--threshold', 0.5,
            '--weights', 'imagenet'], LOG_DIR / f'{CLASSIFIER_DIR.name}.log')
(RUN_ROOT / 'latest_classifier.txt').write_text(CLASSIFIER_DIR.name, encoding='utf-8')
print('분류기 결과:', CLASSIFIER_DIR)


## 10. 결과 확인과 팀에 공유할 ZIP 만들기

`metrics.csv`는 정상 오탐(FP), 언더필 미탐(FN), 재현율과 AUROC를 담습니다. 정확도만 보지 말고 실제 사진의 오탐·미탐을 함께 확인해 주세요. 이번에는 하나의 출력에서 시간으로 나눈 1회 예비 결과이므로 새 출력 일반화나 유의한 성능 개선을 확정할 수 없습니다.

다운로드 ZIP에는 생성 50장, 실제 50장 확인판, 비슷한 학습 사진 짝, 분할·경로 목록, 로그, 예측과 지표를 넣습니다. 용량을 줄이기 위해 무거운 **모델 체크포인트와 전체 시험 이미지**는 ZIP에서 빼고 Drive 결과 폴더에 보관합니다.


In [ ]:
import pandas as pd
metrics = pd.read_csv(CLASSIFIER_DIR / 'metrics.csv')
display(metrics)
attempt = CLASSIFIER_DIR.name
RESULT_ZIP = RUN_ROOT / f'{RUN_NAME}_{attempt}_results.zip'
if RESULT_ZIP.exists():
    raise FileExistsError(f'기존 결과 ZIP 보호: {RESULT_ZIP}. 새 이름을 사용하세요.')
include = []
for root in (PREPARED, GAN_DIR, CLASSIFIER_DIR, PREVIEW_DIR, LOG_DIR):
    for path in root.rglob('*'):
        if not path.is_file():
            continue
        rel = path.relative_to(RUN_ROOT)
        if root == PREPARED and rel.parts[:3] != ('prepared', 'data', 'train_real') and 'data' in rel.parts:
            continue
        if root == PREPARED and 'data' in rel.parts and 'underfill' not in rel.parts:
            continue
        if path.suffix.lower() in {'.pt', '.pth', '.tmp'}:
            continue
        include.append(path)
include += [RUN_ROOT / 'run_lock.json', RUN_ROOT / 'latest_classifier.txt']
with zipfile.ZipFile(RESULT_ZIP, 'x', compression=zipfile.ZIP_DEFLATED, compresslevel=4) as archive:
    for path in sorted(set(include)):
        archive.write(path, arcname=path.relative_to(RUN_ROOT).as_posix())
print('공유할 결과 ZIP:', RESULT_ZIP)
print('크기(MB):', round(RESULT_ZIP.stat().st_size / 1024 / 1024, 1))
print('체크포인트는 Drive의 GAN 및 classifier 폴더에 별도로 보관됩니다.')
if AUTO_DOWNLOAD:
    from google.colab import files
    files.download(str(RESULT_ZIP))


## 11. 종료

1. 결과 ZIP을 내려받아 공유해 주세요. 다운로드가 시작되지 않으면 위에 표시된 Drive 경로에서 직접 내려받을 수 있습니다.
2. 생성 이미지 50장과 `nearest_pairs_*.png`를 살펴봐 주세요. 자동 선택한 실사 50장의 마스크도 사람 검토가 필요합니다.
3. 다음에도 이어서 작업할 필요가 없다면 **런타임 → 세션 연결 해제 및 삭제**로 GPU를 반납해 주세요. Drive에 저장한 결과는 남습니다.
